# Evaluation notebook: CNN vs ViT on long-tailed datasets



In [ ]:
import os
# Evaluation outputs go to their own folder so the training results are never overwritten.
os.environ.setdefault('LT_OUTPUT_ROOT', 'working/evaluation_results')

## 1. Configuration

Shared with the training notebook, copied unchanged. Training-only values such as `EPOCHS` are unused here.

In [ ]:
import os, json, random, time, gc, math, copy, pickle, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
from scipy import sparse

# All input data are local. No downloads or pretrained weights are requested.
INPUT_ROOT = Path(os.environ.get('LT_INPUT_ROOT', 'input'))
OUTPUT_ROOT = Path(os.environ.get('LT_OUTPUT_ROOT', 'working/long_tail_results'))
SMOKE_TEST = os.environ.get('LT_SMOKE_TEST', '0') == '1'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
EPOCHS = 1 if SMOKE_TEST else 50  # baseline budget, not the DeiT-LT paper schedule
BATCH_SIZE = 2 if SMOKE_TEST else 64
NUM_WORKERS = 0 if SMOKE_TEST else min(4, os.cpu_count() or 1)
USE_AMP = DEVICE.type == 'cuda'
RESUME = True
CHECKPOINT_EVERY = 1
MAX_TRAIN_BATCHES = 1 if SMOKE_TEST else None
MAX_EVAL_BATCHES = 2 if SMOKE_TEST else None
VIT_IMAGE_SIZE = 32 if SMOKE_TEST else 224
VIT_PATCH_SIZE = 4 if SMOKE_TEST else 16
LARGE_CNN_IMAGE_SIZE = 32 if SMOKE_TEST else 224

# Each entry can override root, image_root, train, val and test paths.
# Example: DATASET_PATHS['imagenet_lt'] = {
#   'root': '/kaggle/input/my-bundle/imagenet_lt',
#   'image_root': '/kaggle/input/my-bundle/imagenet',
#   'train': '/kaggle/input/my-bundle/splits/ImageNet_LT_train.txt',
#   'val': '/kaggle/input/my-bundle/splits/ImageNet_LT_val.txt',
#   'test': '/kaggle/input/my-bundle/splits/ImageNet_LT_test.txt'}
DATASET_PATHS = {'cifar10_lt': {}, 'cifar100_lt': {},
                 'imagenet_lt': {}, 'inaturalist2018': {}}
SOURCES = {
    # 'cifar10_lt': 'pankrzysiu/cifar10-python',
    # 'cifar100_lt': 'fedesoriano/cifar100',
    'imagenet_lt': 'chanplusplus/imagenetlt',
    'inaturalist2018': None,  # No complete public Kaggle mirror verified.
}
# Attach the three sources above with Kaggle's Add Input control.
# Defaults apply only when the verified source layout is mounted.
_source_suffixes = {
    # 'cifar10_lt': Path('cifar-10-batches-py'),
    # 'cifar100_lt': Path('.'),
    'imagenet_lt': Path('torch_image_folder/mnt/volume_sfo3_01/imagenet-lt/ImageDataset'),
}
for _key, _ref in SOURCES.items():
    if not _ref:
        continue
    _owner, _slug = _ref.split('/')
    # Kaggle supports both legacy flat mounts and owner-qualified dataset mounts.
    _mounts = [INPUT_ROOT / _slug, INPUT_ROOT / 'datasets' / _owner / _slug,
               INPUT_ROOT / _owner / _slug]
    for _mount in _mounts:
        _root = _mount / _source_suffixes[_key]
        if _root.is_dir():
            DATASET_PATHS[_key]['root'] = str(_root)
            if _key == 'imagenet_lt':
                DATASET_PATHS[_key].update(prefer_folders=True, manifest_root=str(_mount))
            break

# Require the three chosen benchmarks. iNaturalist-2018 is excluded at the user's request.
ALLOW_MISSING_DATASETS = False
DATASET_META = {
    'cifar10_lt': {'classes': 10, 'title': 'CIFAR-10-LT'},
    'cifar100_lt': {'classes': 100, 'title': 'CIFAR-100-LT'},
    'imagenet_lt': {'classes': 1000, 'title': 'ImageNet-LT'},
    'inaturalist2018': {'classes': 8142, 'title': 'iNaturalist-2018'},
}
# The paper evaluates both CIFAR datasets at rho=100 and rho=50.
EXPERIMENTS = [
    # {'dataset': 'cifar10_lt', 'imbalance_ratio': 100},
    # {'dataset': 'cifar10_lt', 'imbalance_ratio': 50},
    # {'dataset': 'cifar100_lt', 'imbalance_ratio': 100},
    # {'dataset': 'cifar100_lt', 'imbalance_ratio': 50},
    {'dataset': 'imagenet_lt', 'imbalance_ratio': None},
]
# To add iNaturalist-2018 later, attach its correct 2018 image data, fill
# DATASET_PATHS['inaturalist2018'], and append:
# EXPERIMENTS.append({'dataset': 'inaturalist2018', 'imbalance_ratio': None})

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
if SMOKE_TEST:
    torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({'device': str(DEVICE), 'torch': torch.__version__,
       'torchvision': torchvision.__version__, 'input_root': str(INPUT_ROOT),
       'experiments': len(EXPERIMENTS), 'smoke_test': SMOKE_TEST})

In [ ]:
# ---- Evaluation settings -------------------------------------------------------------
EVAL_RECIPE = 'cnn_vs_vit'          # must equal NOTEBOOK_RECIPE used by the training notebook
MODEL_NAMES = ['cnn', 'vit']        # models to evaluate; missing runs are skipped and reported
EVAL_SPLITS = ['test']              # add 'val' to also score the selection split (optimistic: it picked best.pt)
CKPT_ROOT = os.environ.get('LT_CKPT_ROOT')   # folder containing <EVAL_RECIPE>/...; None = auto-detect
MATCH_TOLERANCE = 1e-3              # re-evaluated top-1 vs the training run's final_metrics.json
ROUND_DECIMALS = 3                  # metric precision in evaluation_summary.csv (None = full precision)
MODEL_LABELS = {}                   # optional display names for the 'model' column, e.g. {'vit': 'deit_tiny'}

## 2. Dataset discovery, loading and preflight

Identical to the training notebook, so the evaluation images and labels are exactly the same.

In [ ]:
# Discovery searches metadata directories, not millions of individual image paths.
ALIASES = {
    # 'cifar10_lt': ['cifar10', 'cifar-10', 'cifar10_lt', 'cifar-10-lt'],
    # 'cifar100_lt': ['cifar100', 'cifar-100', 'cifar100_lt', 'cifar-100-lt'],
    'imagenet_lt': ['imagenet_lt', 'imagenet-lt', 'imagenetlt'],
    # 'inaturalist2018': ['inaturalist2018', 'inaturalist-2018', 'inaturalist18', 'inat2018', 'inat18', 'inat-2018'],
}

def normalized(value):
    return ''.join(ch for ch in str(value).lower() if ch.isalnum())

def metadata_inventory(root, max_depth=5):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f'Input directory does not exist: {root}. Attach the Kaggle dataset or set INPUT_ROOT.')
    directories, files = [root], []
    # Keep train/val/test parents for folder-format datasets, but avoid their images.
    image_dirs = {'images', 'imgs', 'jpegimages', 'train_val2018', 'train', 'val', 'test'}
    for current, dirs, names in os.walk(root):
        current = Path(current)
        depth = len(current.relative_to(root).parts)
        if depth >= max_depth or (depth > 0 and current.name.lower() in image_dirs):
            dirs[:] = []
        else:
            dirs[:] = sorted(d for d in dirs if not d.startswith('.'))
        directories.extend(current / d for d in dirs)
        files.extend(current / name for name in sorted(names)
                     if name.lower().endswith(('.txt', '.json'))
                     or name in {'data_batch_1', 'train', 'test_batch', 'meta'})
    return sorted(set(directories)), sorted(files)

DIRECTORIES, METADATA_FILES = metadata_inventory(INPUT_ROOT)

def choose_unique(paths, description):
    paths = sorted(set(Path(p) for p in paths))
    if len(paths) > 1:
        raise ValueError(f'Ambiguous {description}: {[str(p) for p in paths]}. Set DATASET_PATHS explicitly.')
    return paths[0] if paths else None

def find_named_file(key, split, root=None):
    override = DATASET_PATHS[key].get(split)
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    named = []
    for alias in ALIASES[key]:
        for ext in ['txt', 'json']:
            named.extend([normalized(f'{alias}_{split}.{ext}'),
                          normalized(f'{split}_{alias}.{ext}')])
    if key == 'inaturalist2018':
        named.extend([normalized(f'{split}2018.json'), normalized(f'{split}2018.txt')])
    if root is not None:
        # Read explicit roots directly as they may sit beyond the inventory depth
        # or be mounted through symlinks that os.walk deliberately does not follow.
        root = Path(root)
        local = [p for p in root.iterdir() if p.is_file()
                 and (normalized(p.name) in named
                      or p.name.lower() in {f'{split}.txt', f'{split}.json'})]
        if local:
            return choose_unique(local, f'{key} {split} manifest in {root}')
    candidates = [p for p in METADATA_FILES if normalized(p.name) in named]
    if root is not None:
        candidates.extend(p for p in METADATA_FILES if p.parent == root
                          and p.name.lower() in {f'{split}.txt', f'{split}.json'})
    return choose_unique(candidates, f'{key} {split} manifest')

def find_dataset_root(key):
    override = DATASET_PATHS[key].get('root')
    if override:
        root = Path(override)
        if not root.is_dir():
            raise FileNotFoundError(root)
        return root
    candidates = [d for d in DIRECTORIES if normalized(d.name) in
                  {normalized(a) for a in ALIASES[key]}]
    if candidates:
        return choose_unique(candidates, f'{key} root')
    return None

class ArrayImages(Dataset):
    def __init__(self, data, targets):
        self.data = data
        self.targets = np.asarray(targets, dtype=np.int64)
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, index):
        return Image.fromarray(self.data[index]).convert('RGB'), int(self.targets[index])

class FileImages(Dataset):
    def __init__(self, paths, targets):
        self.paths = list(map(Path, paths))
        self.targets = np.asarray(targets, dtype=np.int64)
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, index):
        try:
            with Image.open(self.paths[index]) as image:
                return image.convert('RGB'), int(self.targets[index])
        except Exception as exc:
            raise RuntimeError(f'Cannot decode image {self.paths[index]}') from exc

class TransformedSubset(Dataset):
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.transform = base, np.asarray(indices), transform
        self.targets = np.asarray(base.targets)[self.indices]
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, index):
        image, label = self.base[int(self.indices[index])]
        return self.transform(image), label

def read_cifar_raw(key):
    # Official extracted Python archives, loaded without downloading or writing to /kaggle/input.
    cname = 'cifar-10-batches-py' if key == 'cifar10_lt' else 'cifar-100-python'
    override = find_dataset_root(key)
    candidates = [d for d in DIRECTORIES if d.name == cname]
    if override:
        candidates = [d for d in [override, override / cname]
                      if (d / ('data_batch_1' if key == 'cifar10_lt' else 'train')).is_file()]
    root = choose_unique(candidates, f'{key} extracted Python archive')
    if root is None:
        return None
    def batch(names):
        arrays, labels = [], []
        for name in names:
            with open(root / name, 'rb') as handle:
                obj = pickle.load(handle, encoding='latin1')
            arrays.append(np.asarray(obj['data']).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1))
            labels.extend(obj['labels'] if key == 'cifar10_lt' else obj['fine_labels'])
        return ArrayImages(np.concatenate(arrays), labels)
    train_names = [f'data_batch_{i}' for i in range(1, 6)] if key == 'cifar10_lt' else ['train']
    test_names = ['test_batch'] if key == 'cifar10_lt' else ['test']
    return batch(train_names), None, batch(test_names), {'format': 'official-cifar-python', 'root': str(root)}

def manifest_entries(path, category_map=None):
    if path.suffix.lower() == '.json':
        with open(path) as handle:
            obj = json.load(handle)
        if not isinstance(obj, dict) or 'images' not in obj or 'annotations' not in obj:
            raise ValueError(f'{path}: expected COCO/iNaturalist images and annotations arrays.')
        if category_map is None:
            ids = sorted(c['id'] for c in obj.get('categories', []))
            if not ids:
                ids = sorted({a['category_id'] for a in obj['annotations']})
            category_map = {category: i for i, category in enumerate(ids)}
        labels = {}
        for ann in obj['annotations']:
            image_id, category = ann['image_id'], ann['category_id']
            if category not in category_map:
                raise ValueError(f'Unknown category {category} in {path}')
            if image_id in labels and labels[image_id] != category_map[category]:
                raise ValueError(f'Conflicting labels for image {image_id} in {path}')
            labels[image_id] = category_map[category]
        entries = [(i['file_name'], labels[i['id']]) for i in obj['images'] if i['id'] in labels]
        if len(entries) != len(obj['images']):
            raise ValueError(f'{path}: unlabeled images cannot be used for evaluation.')
        return entries, category_map
    entries = []
    for line_number, line in enumerate(path.read_text().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        try:
            filename, label = line.rsplit(None, 1)
            entries.append((filename, int(label)))
        except Exception as exc:
            raise ValueError(f'{path}:{line_number}: expected relative/image/path label') from exc
    return entries, category_map

def resolve_manifest_images(key, manifest, entries, dataset_root):
    if not entries:
        raise ValueError(f'Empty manifest: {manifest}')
    configured = DATASET_PATHS[key].get('image_root')
    roots = [Path(configured)] if configured else []
    if not configured:
        roots += [dataset_root, manifest.parent, manifest.parent.parent, INPUT_ROOT]
        roots += [d for d in DIRECTORIES if normalized(d.name) in
                  {'images', 'imgs', 'imagenet', 'imagenet1k', 'ilsvrc2012', 'trainval2018',
                   'inaturalist2018', 'inat2018'}]
        for parent in [dataset_root, manifest.parent, manifest.parent.parent]:
            if parent:
                roots.extend(parent / child for child in ['images', 'data', 'train_val2018'])
    roots = list(dict.fromkeys(Path(p) for p in roots if p is not None))
    sample = entries[:min(16, len(entries))]
    # Some bundles add/remove an ImageNet/ or iNaturalist/ prefix.
    for trim in range(4):
        for root in roots:
            def location(filename):
                p = Path(filename.replace('\\', '/'))
                if p.is_absolute():
                    return p
                return root.joinpath(*p.parts[trim:])
            if all(location(name).is_file() for name, _ in sample):
                return [location(name) for name, _ in entries], {'image_root': str(root), 'trim_components': trim}
    raise FileNotFoundError(f'Cannot resolve images in {manifest}. First path: {entries[0][0]}. '
                            f'Set DATASET_PATHS[{key!r}]["image_root"] to the extracted image directory.')

def folder_dataset(path, class_to_idx=None):
    ds = torchvision.datasets.ImageFolder(str(path))
    if class_to_idx is None:
        # Numeric folder names carry the original class IDs. Lexical ordering
        # would map 10 to class 2 and corrupt comparisons with official manifests.
        if all(name.isdigit() for name in ds.classes):
            class_to_idx = {name: int(name) for name in ds.classes}
        else:
            class_to_idx = ds.class_to_idx
    elif not set(ds.classes) <= set(class_to_idx):
        raise ValueError(f'{path}: evaluation has unknown class folders.')
    paths = [p for p, _ in ds.samples]
    targets = [class_to_idx[ds.classes[label]] for _, label in ds.samples]
    return FileImages(paths, targets), class_to_idx

def read_verified_kaggle_folders(key, root):
    train, mapping = folder_dataset(root / 'train')
    val = folder_dataset(root / 'val', mapping)[0] if (root / 'val').is_dir() else None
    test = folder_dataset(root / 'test', mapping)[0] if (root / 'test').is_dir() else None
    if test is None:
        raise FileNotFoundError(f'{root}: missing final test class folders.')
    manifest_root = Path(DATASET_PATHS[key]['manifest_root'])
    checks = {}
    for split, dataset in [('train', train), ('val', val), ('test', test)]:
        if dataset is None:
            continue
        manifest = find_named_file(key, split, manifest_root)
        if manifest is None:
            raise FileNotFoundError(f'{key}: missing {split} manifest for Kaggle folder verification.')
        entries, _ = manifest_entries(manifest)
        expected = Counter(label for _, label in entries)
        actual = Counter(map(int, dataset.targets))
        if expected != actual:
            raise ValueError(f'{key}/{split}: numeric class folders disagree with official manifest class counts.')
        checks[split] = {'manifest': str(manifest), 'images': len(dataset), 'per_class_counts_match': True}
    return train, val, test, {'format': 'kaggle-numeric-class-folders', 'root': str(root),
                              'source': KAGGLE_SOURCES[key], 'manifest_checks': checks}

def read_dataset_sources(key):
    if key.startswith('cifar'):
        raw = read_cifar_raw(key)
        if raw:
            return raw
    root = find_dataset_root(key)
    if key == 'imagenet_lt' and root and not DATASET_PATHS[key].get('image_root'):
        # Detect the verified renamed-image layout independently of the config
        # presets, including when a user supplies the dataset mount as root.
        folder_root = root / 'torch_image_folder/mnt/volume_sfo3_01/imagenet-lt/ImageDataset'
        if folder_root.is_dir():
            DATASET_PATHS[key].update(root=str(folder_root), prefer_folders=True,
                                     manifest_root=str(root))
            root = folder_root
    if root and DATASET_PATHS[key].get('prefer_folders'):
        return read_verified_kaggle_folders(key, root)
    train_file = find_named_file(key, 'train', root)
    val_file = find_named_file(key, 'val', root)
    test_file = find_named_file(key, 'test', root)
    if key == 'inaturalist2018' and test_file and not DATASET_PATHS[key].get('test'):
        # The public competition test JSON is often included but has no labels.
        # Use the official labeled val2018 split as final evaluation instead.
        if test_file.suffix.lower() == '.json':
            with open(test_file) as handle:
                test_object = json.load(handle)
            if not test_object.get('annotations'):
                test_file = None
    if train_file:
        source, mapping, details = {}, None, {'format': 'manifest', 'manifests': {}}
        for split, path in [('train', train_file), ('val', val_file), ('test', test_file)]:
            if path is None:
                source[split] = None
                continue
            entries, mapping = manifest_entries(path, mapping)
            paths, layout = resolve_manifest_images(key, path, entries, root)
            source[split] = FileImages(paths, [label for _, label in entries])
            details['manifests'][split] = {'path': str(path), **layout}
        if source['test'] is None:
            # iNat 2018 has a labeled validation set and an unlabeled competition test set.
            # A lone labeled val split is final evaluation, never checkpoint selection.
            source['test'], source['val'] = source['val'], None
        if source['test'] is None:
            raise FileNotFoundError(f'{key}: a labeled test/val manifest is required.')
        if source['val'] is not None:
            val_paths, test_paths = set(source['val'].paths), set(source['test'].paths)
            if val_paths & test_paths:
                raise ValueError(f'{key}: validation and test images overlap.')
        for split in ['val', 'test']:
            if source[split] is not None and set(source['train'].paths) & set(source[split].paths):
                raise ValueError(f'{key}: train and {split} images overlap.')
        return source['train'], source['val'], source['test'], details
    if root and (root / 'train').is_dir():
        train, mapping = folder_dataset(root / 'train')
        val = folder_dataset(root / 'val', mapping)[0] if (root / 'val').is_dir() else None
        test = folder_dataset(root / 'test', mapping)[0] if (root / 'test').is_dir() else None
        if test is None:
            test, val = val, None
        if test is not None:
            return train, val, test, {'format': 'class-folders', 'root': str(root)}
    raise FileNotFoundError(f'{key}: no supported data found. Provide extracted CIFAR Python files, '
                            'labeled manifests, or dataset/train/<class> and dataset/test/<class> folders. '
                            'ImageNet-LT requires the official LT split, not balanced ImageNet training data.')

def long_tail_indices(targets, num_classes, imbalance_ratio, seed=SEED):
    labels = np.asarray(targets)
    counts = np.bincount(labels, minlength=num_classes)
    if np.any(counts == 0):
        raise ValueError('Every training class must be present before long-tail sampling.')
    if counts.max() != counts.min():
        # Never silently long-tail an already imbalanced dataset a second time.
        observed = counts.max() / counts.min()
        if not np.isclose(observed, imbalance_ratio, rtol=0.05):
            raise ValueError(f'Already imbalanced CIFAR data have rho={observed:.2f}, requested {imbalance_ratio}. '
                             'Supply balanced CIFAR or a matching pre-made LT split.')
        return np.arange(len(labels)), counts, 'pre-existing LT split'
    sizes = np.array([max(1, int(counts.max() * (1 / imbalance_ratio) ** (c / (num_classes - 1))))
                      for c in range(num_classes)])
    rng = np.random.default_rng(seed)
    selected = np.concatenate([rng.permutation(np.flatnonzero(labels == c))[:sizes[c]]
                               for c in range(num_classes)])
    rng.shuffle(selected)
    return selected, sizes, 'exponential sampling from balanced CIFAR'

def shot_groups(key, counts):
    if key == 'cifar10_lt':
        # Supplement A.1: first 3 / next 4 / last 3 classes.
        return {'many': np.arange(0, 3), 'medium': np.arange(3, 7), 'few': np.arange(7, 10)}
    if key == 'cifar100_lt':
        # Supplement A.1: first 36 / next 35 / final 29 classes.
        return {'many': np.arange(0, 36), 'medium': np.arange(36, 71), 'few': np.arange(71, 100)}
    return {'many': np.flatnonzero(counts > 100),
            'medium': np.flatnonzero((counts >= 20) & (counts <= 100)),
            'few': np.flatnonzero(counts < 20)}

def prepare_experiment(experiment):
    key, ratio = experiment['dataset'], experiment['imbalance_ratio']
    nc = DATASET_META[key]['classes']
    train, val, test, details = read_dataset_sources(key)
    for split, dataset in [('train', train), ('val', val), ('test', test)]:
        if dataset is not None:
            labels = np.asarray(dataset.targets)
            if len(labels) == 0 or labels.min() < 0 or labels.max() >= nc:
                raise ValueError(f'{key}/{split}: labels must be 0..{nc-1}, and the split must be nonempty.')
    if key.startswith('cifar'):
        indices, counts, sampling = long_tail_indices(train.targets, nc, ratio)
    else:
        indices = np.arange(len(train))
        counts = np.bincount(train.targets, minlength=nc)
        sampling = 'official long-tail split, unchanged'
        if not SMOKE_TEST and (np.any(counts == 0) or counts.max() == counts.min()):
            raise ValueError(f'{key}: expected a long-tailed training split covering all {nc} classes.')
        expected_size = {'imagenet_lt': 115846, 'inaturalist2018': 437513}[key]
        if not SMOKE_TEST and len(train) != expected_size:
            warnings.warn(f'{key}: found {len(train):,} training images; the paper reports '
                          f'{expected_size:,}. Verify that the supplied split is the intended benchmark.')
    if not SMOKE_TEST and np.unique(test.targets).size != nc:
        raise ValueError(f'{key}: labeled evaluation must cover all {nc} classes.')
    tag = key + (f'_if{ratio}' if ratio else '')
    return {'key': key, 'tag': tag, 'num_classes': nc, 'train': train, 'val': val, 'test': test,
            'train_indices': indices, 'counts': counts, 'groups': shot_groups(key, counts),
            'details': {**details, 'sampling': sampling, 'imbalance_ratio': ratio}}

# Fail early before starting a long run if any benchmark is missing.
def preflight():
    global ACTIVE_EXPERIMENTS
    rows, missing = [], []
    ACTIVE_EXPERIMENTS = []
    for exp in EXPERIMENTS:
        try:
            data = prepare_experiment(exp)
        except FileNotFoundError as exc:
            if not ALLOW_MISSING_DATASETS:
                raise
            missing.append({'dataset': exp['dataset'], 'imbalance_ratio': exp['imbalance_ratio'],
                            'reason': str(exc)})
            continue
        for base, indices in [(data['train'], data['train_indices']),
                              (data['test'], np.arange(len(data['test'])))]:
            for i in indices[np.linspace(0, len(indices)-1, min(8, len(indices)), dtype=int)]:
                image, label = base[int(i)]
                assert image.mode == 'RGB' and 0 <= label < data['num_classes']
        rows.append({'experiment': data['tag'], 'classes': data['num_classes'],
                     'selection_val': len(data['val']) if data['val'] is not None else 0,
                     'final_eval': len(data['test']), 'head_max': int(data['counts'].max()),
                     'tail_min': int(data['counts'].min()), 'format': data['details']['format']})
        ACTIVE_EXPERIMENTS.append(exp)
        del data
        gc.collect()
    if missing:
        print('MISSING BENCHMARKS: these will not run and no results will be reported for them.')
        display(pd.DataFrame(missing))
        pd.DataFrame(missing).to_csv(OUTPUT_ROOT / 'missing_datasets.csv', index=False)
    else:
        (OUTPUT_ROOT / 'missing_datasets.csv').unlink(missing_ok=True)
    if not ACTIVE_EXPERIMENTS:
        raise FileNotFoundError('No requested dataset is attached. Add the Kaggle sources listed in KAGGLE_SOURCES.')
    return pd.DataFrame(rows)

dataset_summary = preflight()
display(dataset_summary)
dataset_summary.to_csv(OUTPUT_ROOT / 'dataset_inventory.csv', index=False)

## 3. Model definitions

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNetCIFAR(nn.Module):
    # ResNet-32 = 3 stages x 5 blocks, He et al. CIFAR variant
    def __init__(self, num_blocks=(5,5,5), num_classes=100):
        super().__init__()
        self.in_planes = 16
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, num_blocks[0], 1)
        self.layer2 = self._make_layer(32, num_blocks[1], 2)
        self.layer3 = self._make_layer(64, num_blocks[2], 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)
    def _make_layer(self, planes, n, stride):
        strides = [stride] + [1]*(n-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out); out = self.layer2(out); out = self.layer3(out)
        out = self.avgpool(out).flatten(1)
        return self.fc(out)

def resnet32(num_classes=100):
    return ResNetCIFAR((5,5,5), num_classes)

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, embed_dim=192):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x)               # B, D, H/P, W/P
        return x.flatten(2).transpose(1, 2)  # B, N, D

class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, num_classes=100,
                 embed_dim=192, depth=8, n_heads=3, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_ch, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(drop)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=int(embed_dim*mlp_ratio),
            dropout=drop, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.pos_drop(x)
        x = self.encoder(x)
        x = self.norm(x)
        return self.head(x[:, 0])

def vit_tiny(num_classes=100):
    return ViT(num_classes=num_classes, embed_dim=192, depth=8, n_heads=3)

## 4. Transforms, loaders and memory-safe evaluation

The training notebook's helper cell, cut off before `train_and_evaluate`. Nothing here trains.

In [ ]:
class ThreeAugment:
    def __init__(self):
        self.ops = [T.RandomGrayscale(p=1.0), T.RandomSolarize(threshold=128, p=1.0),
                    T.GaussianBlur(kernel_size=3)]
    def __call__(self, image):
        return random.choice(self.ops)(image)

def make_loaders(data, model_name):
    cifar_cnn = data['key'].startswith('cifar') and model_name == 'cnn'
    image_size = 32 if cifar_cnn else (LARGE_CNN_IMAGE_SIZE if model_name == 'cnn' else VIT_IMAGE_SIZE)
    if data['key'] == 'cifar10_lt':
        mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
    elif data['key'] == 'cifar100_lt':
        mean, std = (0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)
    else:
        mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    crop = T.RandomCrop(32, padding=4) if cifar_cnn else T.RandomResizedCrop(image_size, scale=(0.08, 1.0))
    ops = [crop, T.RandomHorizontalFlip()]
    if model_name == 'deit3':
        ops += [ThreeAugment(), T.ColorJitter(0.3, 0.3, 0.3)]
    train_tf = T.Compose(ops + [T.ToTensor(), T.Normalize(mean, std)])
    eval_ops = [] if cifar_cnn else [T.Resize(int(image_size / 0.875)), T.CenterCrop(image_size)]
    test_tf = T.Compose(eval_ops + [T.ToTensor(), T.Normalize(mean, std)])
    train_ds = TransformedSubset(data['train'], data['train_indices'], train_tf)
    val_ds = (TransformedSubset(data['val'], np.arange(len(data['val'])), test_tf)
              if data['val'] is not None else None)
    test_ds = TransformedSubset(data['test'], np.arange(len(data['test'])), test_tf)
    generator = torch.Generator().manual_seed(SEED)
    def seed_worker(worker_id):
        seed = torch.initial_seed() % (2**32)
        np.random.seed(seed)
        random.seed(seed)
    common = {'num_workers': NUM_WORKERS, 'pin_memory': DEVICE.type == 'cuda',
              'worker_init_fn': seed_worker}
    loaders = {
        # A singleton final batch breaks CNN BatchNorm for tiny smoke fixtures.
        'train': DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                            drop_last=len(train_ds) > BATCH_SIZE,
                            generator=generator, **common),
        'val': DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **common) if val_ds else None,
        'test': DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **common),
    }
    return loaders, image_size

def make_model(name, num_classes, image_size, dataset_key):
    if name == 'cnn':
        if dataset_key.startswith('cifar') or SMOKE_TEST:
            return ResNetCIFAR((1, 1, 1) if SMOKE_TEST else (5, 5, 5), num_classes)
        return torchvision.models.resnet50(weights=None, num_classes=num_classes)
    kwargs = {'img_size': image_size, 'patch_size': VIT_PATCH_SIZE,
              'num_classes': num_classes, 'embed_dim': 24 if SMOKE_TEST else 192,
              'depth': 1 if SMOKE_TEST else 8, 'n_heads': 3}
    if name == 'vit':
        return ViT(**kwargs)
    if name == 'deit3':
        return DeiT3(**kwargs, drop_path_max=0.1, layerscale_init=1e-4)
    raise ValueError(name)

def classification_loss(logits, labels, mode, train=False):
    if mode == 'bce':
        return F.binary_cross_entropy_with_logits(logits, F.one_hot(labels, logits.size(1)).float())
    return F.cross_entropy(logits, labels, label_smoothing=0.1 if train and mode == 'smooth_ce' else 0.0)

def safe_group_mean(array, group):
    values = array[group]
    finite = values[np.isfinite(values)]
    return float(finite.mean()) if len(finite) else None

@torch.no_grad()
def evaluate(model, loader, data, loss_mode):
    # Stream sufficient statistics. Never retain an N x 8142 matrix of logits.
    model.eval()
    nc = data['num_classes']
    support = np.zeros(nc, dtype=np.int64)
    predicted_count = np.zeros(nc, dtype=np.int64)
    true_positive = np.zeros(nc, dtype=np.int64)
    labels_all, preds_all = [], []
    loss_sum, top5_count, total = 0.0, 0, 0
    for batch_idx, (images, labels) in enumerate(loader):
        if MAX_EVAL_BATCHES is not None and batch_idx >= MAX_EVAL_BATCHES:
            break
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
            loss = classification_loss(logits, labels, loss_mode)
        predictions = logits.argmax(1)
        top5_count += (logits.topk(min(5, nc), dim=1).indices == labels[:, None]).any(1).sum().item()
        ys, ps = labels.cpu().numpy(), predictions.cpu().numpy()
        support += np.bincount(ys, minlength=nc)
        predicted_count += np.bincount(ps, minlength=nc)
        true_positive += np.bincount(ys[ys == ps], minlength=nc)
        labels_all.append(ys)
        preds_all.append(ps)
        total += len(ys)
        loss_sum += float(loss) * len(ys)
    if not total:
        raise RuntimeError('Evaluation received no batches.')
    per_class_acc = np.divide(true_positive, support, out=np.full(nc, np.nan), where=support > 0)
    precision = np.divide(true_positive, predicted_count, out=np.zeros(nc, dtype=float), where=predicted_count > 0)
    recall = np.divide(true_positive, support, out=np.zeros(nc, dtype=float), where=support > 0)
    denom = support + predicted_count
    f1 = np.divide(2 * true_positive, denom, out=np.zeros(nc, dtype=float), where=denom > 0)
    ys, ps = np.concatenate(labels_all), np.concatenate(preds_all)
    cm = sparse.coo_matrix((np.ones(total, dtype=np.int64), (ys, ps)), shape=(nc, nc)).tocsr()
    cm.sum_duplicates()
    return {'loss': loss_sum / total, 'top1': float(true_positive.sum() / total),
            'top5': float(top5_count / total), 'macro_f1': float(f1.mean()),
            'weighted_f1': float(np.dot(f1, support) / total),
            'precision_macro': float(precision.mean()), 'recall_macro': float(recall.mean()),
            'group_acc': {g: safe_group_mean(per_class_acc, ids) for g, ids in data['groups'].items()},
            'per_class_acc': per_class_acc, 'support': support, 'confusion_matrix': cm,
            'labels': ys, 'predictions': ps, 'evaluated_samples': total,
            'partial_evaluation': MAX_EVAL_BATCHES is not None}

def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, np.generic):
        return json_ready(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

def atomic_json(path, obj):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(json_ready(obj), indent=2, allow_nan=False))
    temporary.replace(path)

## 5. Load checkpoints, evaluate and write the CSV files

In [ ]:
def find_checkpoint_base():
    # Training layout: <base>/<recipe>/<experiment tag>/<model>_seed<SEED>/{best,final,last}.pt
    if CKPT_ROOT:
        base = Path(CKPT_ROOT)
        if not (base / EVAL_RECIPE).is_dir():
            raise FileNotFoundError(f'{base / EVAL_RECIPE} does not exist. Check CKPT_ROOT / EVAL_RECIPE.')
        return base
    # Same directory as the training notebook first (its long_tail_results folder), then attached inputs.
    candidates = [Path('/kaggle/working/long_tail_results'), Path.cwd() / 'long_tail_results']
    candidates += [d for d in DIRECTORIES if d.name == 'long_tail_results']
    found = sorted({p.resolve() for p in candidates if (p / EVAL_RECIPE).is_dir()})
    if not found:
        raise FileNotFoundError('No training results found. Run this in the training session, or attach the '
                                'training notebook output as an input, or set CKPT_ROOT.')
    if len(found) > 1:
        raise ValueError(f'Several training result folders found: {[str(p) for p in found]}. Set CKPT_ROOT.')
    return found[0]

def load_weights(model, run_dir):
    # best.pt / final.pt are plain state dicts. last.pt is a full training checkpoint written by the
    # training notebook (trusted local file), used only if neither of the others exists.
    for name in ('best.pt', 'final.pt'):
        path = run_dir / name
        if path.is_file():
            model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
            return name
    path = run_dir / 'last.pt'
    if path.is_file():
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
        model.load_state_dict(checkpoint['model'])
        return 'last.pt'
    return None

def read_json(path):
    return json.loads(path.read_text()) if path.is_file() else None

CKPT_BASE = find_checkpoint_base()
print('Reading checkpoints from:', CKPT_BASE / EVAL_RECIPE)

summary_rows, per_class_frames, missing_runs = [], [], []
for experiment in ACTIVE_EXPERIMENTS:
    data = prepare_experiment(experiment)
    group_of = {int(c): g for g, ids in data['groups'].items() for c in ids}
    for model_name in MODEL_NAMES:
        run_dir = CKPT_BASE / EVAL_RECIPE / data['tag'] / f'{model_name}_seed{SEED}'
        if SMOKE_TEST:
            run_dir = run_dir / 'smoke_test'
        if not run_dir.is_dir():
            print(f'SKIP {data["tag"]}/{model_name}: no run folder at {run_dir}')
            missing_runs.append({'experiment': data['tag'], 'model': model_name, 'reason': f'no folder {run_dir}'})
            continue
        loaders, image_size = make_loaders(data, model_name)
        config = read_json(run_dir / 'run_config.json') or {}
        if config and (config.get('num_classes') != data['num_classes'] or config.get('image_size') != image_size):
            raise ValueError(f'{run_dir}: run_config.json (classes={config.get("num_classes")}, '
                             f'image_size={config.get("image_size")}) does not match this notebook '
                             f'(classes={data["num_classes"]}, image_size={image_size}).')
        model = make_model(model_name, data['num_classes'], image_size, data['key']).to(DEVICE)
        checkpoint_name = load_weights(model, run_dir)
        if checkpoint_name is None:
            print(f'SKIP {data["tag"]}/{model_name}: no best.pt, final.pt or last.pt in {run_dir}')
            missing_runs.append({'experiment': data['tag'], 'model': model_name, 'reason': f'no checkpoint in {run_dir}'})
            del model, loaders
            continue
        history = read_json(run_dir / 'history.json') or []
        val_curve = [h['val_top1'] for h in history if 'val_top1' in h]
        loss_mode = 'smooth_ce' if model_name == 'vit' else 'ce'   # same mapping as training (no smoothing at eval)
        n_params = sum(p.numel() for p in model.parameters())
        label = MODEL_LABELS.get(model_name, model_name)
        selection = config.get('selection') or ('validation_top1' if loaders['val'] is not None else 'final_epoch')
        epochs_value = config.get('epochs') or (len(history) or None)
        for split in EVAL_SPLITS:
            loader = loaders.get(split)
            if loader is None:
                print(f'SKIP {data["tag"]}/{model_name}/{split}: this dataset has no {split} split')
                continue
            result = evaluate(model, loader, data, loss_mode)
            matches = None
            reference = read_json(run_dir / 'final_metrics.json')
            if split == 'test' and reference and checkpoint_name in ('best.pt', 'final.pt'):
                matches = bool(abs(reference['top1'] - result['top1']) <= MATCH_TOLERANCE)
            summary_rows.append({
                'experiment': data['tag'], 'dataset': data['key'],
                'imbalance_ratio': data['details']['imbalance_ratio'],
                'model': label, 'split': split, 'seed': SEED,
                'epochs': epochs_value, 'selection': selection, 'run_dir': str(run_dir),
                'checkpoint': checkpoint_name,
                'epochs_planned': config.get('epochs'), 'epochs_trained': len(history) or None,
                'best_val_epoch': (int(np.argmax(val_curve)) + 1) if val_curve else None,
                'best_val_top1': max(val_curve) if val_curve else None,
                'parameters': n_params, 'evaluated_samples': result['evaluated_samples'],
                'loss': result['loss'], 'top1': result['top1'], 'top5': result['top5'],
                'macro_f1': result['macro_f1'], 'weighted_f1': result['weighted_f1'],
                'precision_macro': result['precision_macro'], 'recall_macro': result['recall_macro'],
                'many_acc': result['group_acc'].get('many'), 'medium_acc': result['group_acc'].get('medium'),
                'few_acc': result['group_acc'].get('few'),
                'many_minus_few': (result['group_acc']['many'] - result['group_acc']['few'])
                                  if None not in (result['group_acc'].get('many'), result['group_acc'].get('few')) else None,
                'matches_training_run': matches, 'partial_evaluation': result['partial_evaluation'],
                'smoke_test': SMOKE_TEST})
            per_class_frames.append(pd.DataFrame({
                'experiment': data['tag'], 'model': label, 'split': split,
                'class_id': np.arange(data['num_classes']), 'train_count': data['counts'],
                'eval_count': result['support'], 'accuracy': result['per_class_acc'],
                'group': [group_of.get(i) for i in range(data['num_classes'])]}))
            print(f'{data["tag"]}/{model_name}/{split} [{checkpoint_name}]: top1={result["top1"]:.4f} '
                  f'top5={result["top5"]:.4f} macro_f1={result["macro_f1"]:.4f} '
                  f'many/med/few={result["group_acc"]}', flush=True)
            del result
        del model, loaders
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    del data
    gc.collect()

if not summary_rows:
    raise RuntimeError('Nothing was evaluated. See the SKIP messages above.')

details_df = pd.DataFrame(summary_rows)      # everything, full precision, all evaluated splits
summary_df = details_df                      # (used by the plot cell)
per_class_df = pd.concat(per_class_frames, ignore_index=True)

# Main CSV: same 19 columns, in the same order, as the training notebook's *_all_metrics.csv.
MAIN_COLUMNS = ['dataset', 'experiment', 'model', 'seed', 'epochs', 'selection', 'run_dir', 'loss', 'top1',
                'top5', 'macro_f1', 'weighted_f1', 'precision_macro', 'recall_macro', 'evaluated_samples',
                'partial_evaluation', 'many_acc', 'medium_acc', 'few_acc']
METRIC_COLUMNS = ['loss', 'top1', 'top5', 'macro_f1', 'weighted_f1', 'precision_macro', 'recall_macro',
                  'many_acc', 'medium_acc', 'few_acc']
main_df = details_df[details_df['split'] == 'test'][MAIN_COLUMNS].copy()
if main_df.empty:
    raise RuntimeError("evaluation_summary.csv reports the final evaluation split; put 'test' in EVAL_SPLITS.")
if ROUND_DECIMALS is not None:
    main_df[METRIC_COLUMNS] = main_df[METRIC_COLUMNS].astype(float).round(ROUND_DECIMALS)
main_df['epochs'] = main_df['epochs'].astype('Int64')
main_df['evaluated_samples'] = main_df['evaluated_samples'].astype(int)
main_df.to_csv(OUTPUT_ROOT / 'evaluation_summary.csv', index=False)
details_df.to_csv(OUTPUT_ROOT / 'evaluation_details.csv', index=False)
per_class_df.to_csv(OUTPUT_ROOT / 'evaluation_per_class.csv', index=False)
if missing_runs:
    pd.DataFrame(missing_runs).to_csv(OUTPUT_ROOT / 'evaluation_missing_runs.csv', index=False)
    print('Some requested runs had no checkpoint; see evaluation_missing_runs.csv')
else:
    (OUTPUT_ROOT / 'evaluation_missing_runs.csv').unlink(missing_ok=True)

display(main_df)
if (details_df['matches_training_run'] == False).any():
    print('WARNING: some re-evaluated top-1 values differ from the training run by more than the tolerance.')
if SMOKE_TEST:
    print('SMOKE TEST ONLY: these numbers are not benchmark results.')
print('Saved:', OUTPUT_ROOT / 'evaluation_summary.csv')
print('Saved:', OUTPUT_ROOT / 'evaluation_details.csv')
print('Saved:', OUTPUT_ROOT / 'evaluation_per_class.csv')

## 6. Plots

In [ ]:
# Head / mid / tail accuracy per model, and accuracy against class frequency (the long-tail picture).
for tag, rows in summary_df[summary_df['split'] == 'test'].groupby('experiment', sort=False):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    groups = ['many_acc', 'medium_acc', 'few_acc']
    width = 0.8 / len(rows)
    for i, (_, row) in enumerate(rows.iterrows()):
        values = [np.nan if pd.isna(row[g]) else row[g] for g in groups]
        axes[0].bar(np.arange(3) + (i - (len(rows) - 1) / 2) * width, values, width, label=row['model'])
    axes[0].set_xticks(np.arange(3), ['head', 'mid', 'tail'])
    axes[0].set(title=f'{tag}: test accuracy by class group', ylabel='mean per-class accuracy', ylim=(0, 1))
    axes[0].legend()
    pc = per_class_df[(per_class_df['experiment'] == tag) & (per_class_df['split'] == 'test')]
    for model_name, g in pc.groupby('model', sort=False):
        axes[1].scatter(g['train_count'], g['accuracy'], s=8, alpha=0.5, label=model_name)
    axes[1].set_xscale('log')
    axes[1].set(title='Per-class accuracy vs training images per class', xlabel='training images (log scale)',
                ylabel='accuracy', ylim=(0, 1))
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / f'{EVAL_RECIPE}_{tag}_eval_comparison.png', dpi=150)
    plt.show()
    plt.close(fig)
print('Outputs are in', OUTPUT_ROOT)